# Overview
This notebook prepares a sample of agent skills from the GitSkills dataset for automated security analysis using MalSkills.

# Scope
For initial testing, we will select 100 randomly ordered SKILL.md records from the GitSkills database. This smaller sampler will be used to develop and validate the automated analysis workflow before expanding to a larger dataset.

# Sampling Approach
For this initial experiment, I am randomly selecting 100 SKILL.md files. The SQL query filters for records containing a SKILL.md file with non-null content. The selected files are then exported into indiviudally numbered folders so they can be processed by MalSkills.

For this stage, I will only be exporting SKILL.md files, and not the additional files that may be included in the skill directory.

To keep track of the exported skills, a CSV file (skill_mapping.csv) will be generated. This file maps eaech numbered skill folder to its original GitSkills repo and file path, allowing us to trace any findings from MalSkills back to their original source.

## Connect

Connect to the SQL database prior to running the export script.

In [6]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)  # max characters shown per cell in tables; raise it to see longer text

# Works whether VS Code runs the notebook from notebooks/ or from the project root
ROOT = Path.cwd() if (Path.cwd() / "data" / "sample").exists() else Path.cwd().parent
DB_PATH = ROOT / "data" / "sample" / "agent_skills_sample.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH} (see README)"

# Read-only, so nothing in the database can be changed by accident
conn = sqlite3.connect(f"{DB_PATH.resolve().as_uri()}?mode=ro", uri=True)


# sql("SELECT ...") -> DataFrame. Use ? placeholders for values: sql("... WHERE name = ?", ("pdf",))
def sql(query, params=()):
    return pd.read_sql_query(query, conn, params=params)


print("Connected to", DB_PATH.relative_to(ROOT))

Connected to data\sample\agent_skills_sample.db


## Export Script

In [9]:
# Export multiple skills for linting
from pathlib import Path

# pick 100 random skills from GitSkills

skills_to_export = sql("""
    SELECT repo_full_name, path, content
    FROM artifacts
    WHERE path LIKE '%/SKILL.md'
        AND content IS NOT NULL
    ORDER BY RANDOM()
    LIMIT 100
""")

# export each skill into its own folder
for index, row in skills_to_export.iterrows(): # iterrows(): loop through each row of DataFrame
    # extract repo name, original path, and file contents.
    repo = row["repo_full_name"]
    path = row["path"]
    content = row["content"]

    # unique numbered folder per skill up to 4 numbers.
    path_name = f"skill_{index:04d}"

    # each skill gets its own folder inside exported_skills
    output = Path(f"../exported_skills/{path_name}/SKILL.md")
    output.parent.mkdir(parents=True, exist_ok=True) # create folder if n/a
    output.write_text(content, encoding="utf-8")

    print (f"{path_name}: {repo}/{path}")

print(f"\nExported {len(skills_to_export)} skills.")

# Save a mapping between exported folders and original GitSkills records
skill_mapping = skills_to_export[["repo_full_name", "path"]].copy()

# Generate folder names matching those created during export
skill_mapping["skill_id"] = [f"skill_{index:04d}" for index in skills_to_export.index]

# Organize columns
skill_mapping = skill_mapping[["skill_id", "repo_full_name", "path"]]

# Save mapping to CSV for later analysis
mapping_path = Path("../exported_skills/skill_mapping.csv")
skill_mapping.to_csv(mapping_path, index=False)
print(f"Saved skill mapping to: {mapping_path}")

skill_0000: Mahesh-07-404/Redforge/skills/configuring-identity-aware-proxy-with-google-iap/SKILL.md
skill_0001: MelbourneDeveloper/dart_node/.claude/skills/code-dedup/SKILL.md
skill_0002: nyvlemyt/MultiAgentOS/packages/skills/library/conducting-memory-forensics-with-volatility/SKILL.md
skill_0003: linkerdog/rara/.agents/skills/support-acp/SKILL.md
skill_0004: Trenchant-Cyber-1/trenchant-labs-ai-skills/skills/migrate/SKILL.md
skill_0005: Jono120/useful-ai-skills/Cursor-Skills/dotnet-expert-se/SKILL.md
skill_0006: emrekonakli/argosdemo/.claude/skills/gameobject-set-parent/SKILL.md
skill_0007: retex73/music-app/.claude/skills/dnd-patterns/SKILL.md
skill_0008: trdlabs/lab/.claude/skills/generated/gortex-adapters-platform-5-dirs/SKILL.md
skill_0009: yhzion/claude-plugin-models/plugins/gemini/skills/gemini-cli-runtime/SKILL.md
skill_0010: LongLeo287/OmniClaw/ecosystem/skills/openclaw_worker/SKILL.md
skill_0011: luokai0/ai-agent-skills-by-luo-kai/ai-agent-skills/18-ai-agents-and-automation (b

## Verify the exported files

In [10]:
from pathlib import Path

export_dir = Path("../exported_skills")

# Find all SKILL.md files inside the exported skill folders
exported_files = list(export_dir.glob("skill_*/SKILL.md"))

print(f"Expected files: 100")
print(f"Actual files: {len(exported_files)}")

Expected files: 100
Actual files: 100
